# ODI to Databricks Migration: PRXBI_DW_SEP.WC_BADGE_DETAILS_D_Load

**Conversion Timestamp**: 2024-07-30T12:00:00Z

This notebook migrates an Oracle Data Integrator (ODI) session to Databricks Spark SQL. It processes badge details data, handling staging, flow, and merging into the `WC_BADGE_DETAILS_D` dimension table. It includes deduplication logic, dimension lookups, and flags records for update/insert based on existing data.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "380", "Datasource Num ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

## ETL Parameters

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {1}: Get ETL Job Type
-- SCEN_TASK_NO {2}: Get ETL Last Extract Time
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT MAX(etl_last_extract_time) AS etl_last_extract_time
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

-- SCEN_TASK_NO {3}: Get ETL Current Extract Time
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT MAX(etl_current_extract_time) AS etl_current_extract_time
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

-- SCEN_TASK_NO {4}: Get ETL Last Extract Time (duplicate)
-- SCEN_TASK_NO {5}: Get ETL Current Extract Time (duplicate)
-- SCEN_TASK_NO {6}: Get ROW_WID
CREATE OR REPLACE TEMPORARY VIEW v_etl_row_wid AS
SELECT MAX(ROW_WID) AS row_wid
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
etl_job_type = dbutils.widgets.get("ETL_JOB_TYPE")
datasource_num_id = dbutils.widgets.get("DATASOURCE_NUM_ID")
etl_proc_wid = dbutils.widgets.get("ETL_PROC_WID")
odi_sess_no = dbutils.widgets.get("ODI_SESS_NO")

print(f"ETL_JOB_TYPE: {etl_job_type}")
print(f"DATASOURCE_NUM_ID: {datasource_num_id}")
print(f"ETL_PROC_WID: {etl_proc_wid}")
print(f"ODI_SESS_NO: {odi_sess_no}")

display(spark.sql("SELECT * FROM v_etl_last_extract_time"))
display(spark.sql("SELECT * FROM v_etl_current_extract_time"))
display(spark.sql("SELECT * FROM v_etl_row_wid"))

## Staging Table (C$)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {10}: (No-op)
-- SCEN_TASK_NO {20}: (No-op)
-- SCEN_TASK_NO {30}: Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_badge_staging;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {40}: Create staging table
CREATE TABLE workspace.prxbi_dw.c_badge_staging (
	ID	STRING,
	BADGELOCATION	STRING,
	BADGETOKEN	STRING,
	BADGEVERSION	BIGINT,
	CONTACTEMAIL	STRING,
	CONTACTFIRSTNAME	STRING,
	CONTACTJOBTITLE	STRING,
	CONTACTLASTNAME	STRING,
	CONTACTPERSONRXMASTERID	STRING,
	CREATEDBYREGISTRATIONTYPE	STRING,
	CREATEDBYTYPE	STRING,
	CULTURE	STRING,
	CUSTOMERTYPE	STRING,
	EVENTEDITIONGBSCODE	STRING,
	ISBADGEUPDATE	STRING,
	MARKETINGPREFERENCESPROMPTREQU	STRING,
	ORGANISATIONCITY	STRING,
	ORGANISATIONCOUNTRYCODE	STRING,
	ORGANISATIONDISPLAYNAME	STRING,
	ORGANISATIONRXMASTERID	STRING,
	ORGANISATIONSTATE	STRING,
	PARTICIPATINGORGANISATIONID	STRING,
	PRODUCTCODE	STRING,
	QRCODECONTENT	STRING,
	REGISTRATIONID	STRING,
	STATUS	BIGINT,
	SUPPORTSTAFFCOMPANYADDRESS	STRING,
	SUPPORTSTAFFCOMPANYNAME	STRING,
	SUPPORTSTAFFMOBILEPHONE	STRING,
	SUPPORTSTAFFREPORTSTO	STRING,
	SUPPORTSTAFFSTANDS	STRING,
	SUPPORTSTAFFUSERACCESS	STRING,
	VERSIONNUMBER	BIGINT,
	MOBILEPHONE	STRING,
	FIRSTSCANNEDDATE	TIMESTAMP,
	LASTPRINTEDDATE	TIMESTAMP,
	ACCESSVALIDITYMODIFIEDDATE	TIMESTAMP,
	CREATEDDATE	TIMESTAMP,
	COMPANYPRODUCTCODE	STRING,
	PAYMENTSTATUS	STRING,
	PHOTOKEY	STRING,
	PHOTOSOURCE	STRING,
	PHOTOSOURCETYPE	STRING
) USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {50}: Insert into staging with deduplication (F.12 rewrite)
INSERT INTO workspace.prxbi_dw.c_badge_staging
SELECT	
	ID,
	BADGELOCATION,
	BADGETOKEN,
	BADGEVERSION,
	CONTACTEMAIL,
	CONTACTFIRSTNAME,
	CONTACTJOBTITLE,
	CONTACTLASTNAME,
	CONTACTPERSONRXMASTERID,
	CREATEDBYREGISTRATIONTYPE,
	CREATEDBYTYPE,
	CULTURE,
	CUSTOMERTYPE,
	EVENTEDITIONGBSCODE,
	ISBADGEUPDATE,
	MARKETINGPREFERENCESPROMPTREQU,
	ORGANISATIONCITY,
	ORGANISATIONCOUNTRYCODE,
	ORGANISATIONDISPLAYNAME,
	ORGANISATIONRXMASTERID,
	ORGANISATIONSTATE,
	PARTICIPATINGORGANISATIONID,
	PRODUCTCODE,
	QRCODECONTENT,
	REGISTRATIONID,
	STATUS,
	SUPPORTSTAFFCOMPANYADDRESS,
	SUPPORTSTAFFCOMPANYNAME,
	SUPPORTSTAFFMOBILEPHONE,
	SUPPORTSTAFFREPORTSTO,
	SUPPORTSTAFFSTANDS,
	SUPPORTSTAFFUSERACCESS,
	VERSIONNUMBER,
	MOBILEPHONE,
	FIRSTSCANNEDDATE,
	LASTPRINTEDDATE,
	ACCESSVALIDITYMODIFIEDDATE,
	CREATEDDATE,
	COMPANYPRODUCTCODE,
	PAYMENTSTATUS,
	PHOTOKEY,
	PHOTOSOURCE,
	PHOTOSOURCETYPE
FROM (
    SELECT
        AMERCURY_BADGE_TS.ID,
        AMERCURY_BADGE_TS.BADGELOCATION,
        AMERCURY_BADGE_TS.BADGETOKEN,
        AMERCURY_BADGE_TS.BADGEVERSION,
        AMERCURY_BADGE_TS.CONTACTEMAIL,
        AMERCURY_BADGE_TS.CONTACTFIRSTNAME,
        AMERCURY_BADGE_TS.CONTACTJOBTITLE,
        AMERCURY_BADGE_TS.CONTACTLASTNAME,
        AMERCURY_BADGE_TS.CONTACTPERSONRXMASTERID,
        AMERCURY_BADGE_TS.CREATEDBYREGISTRATIONTYPE,
        AMERCURY_BADGE_TS.CREATEDBYTYPE,
        AMERCURY_BADGE_TS.CULTURE,
        AMERCURY_BADGE_TS.CUSTOMERTYPE,
        AMERCURY_BADGE_TS.EVENTEDITIONGBSCODE,
        AMERCURY_BADGE_TS.ISBADGEUPDATE,
        AMERCURY_BADGE_TS.MARKETINGPREFERENCESPROMPTREQUIRED AS MARKETINGPREFERENCESPROMPTREQU,
        AMERCURY_BADGE_TS.ORGANISATIONCITY,
        AMERCURY_BADGE_TS.ORGANISATIONCOUNTRYCODE,
        AMERCURY_BADGE_TS.ORGANISATIONDISPLAYNAME,
        AMERCURY_BADGE_TS.ORGANISATIONRXMASTERID,
        AMERCURY_BADGE_TS.ORGANISATIONSTATE,
        AMERCURY_BADGE_TS.PARTICIPATINGORGANISATIONID,
        AMERCURY_BADGE_TS.PRODUCTCODE,
        AMERCURY_BADGE_TS.QRCODECONTENT,
        AMERCURY_BADGE_TS.REGISTRATIONID,
        AMERCURY_BADGE_TS.STATUS,
        AMERCURY_BADGE_TS.SUPPORTSTAFFCOMPANYADDRESS,
        AMERCURY_BADGE_TS.SUPPORTSTAFFCOMPANYNAME,
        AMERCURY_BADGE_TS.SUPPORTSTAFFMOBILEPHONE,
        AMERCURY_BADGE_TS.SUPPORTSTAFFREPORTSTO,
        AMERCURY_BADGE_TS.SUPPORTSTAFFSTANDS,
        AMERCURY_BADGE_TS.SUPPORTSTAFFUSERACCESS,
        AMERCURY_BADGE_TS.VERSIONNUMBER,
        AMERCURY_BADGE_TS.MOBILEPHONE,
        AMERCURY_BADGE_TS.FIRSTSCANNEDDATE,
        AMERCURY_BADGE_TS.LASTPRINTEDDATE,
        AMERCURY_BADGE_TS.ACCESSVALIDITYMODIFIEDDATE,
        AMERCURY_BADGE_TS.CREATEDDATE,
        AMERCURY_BADGE_TS.COMPANYPRODUCTCODE,
        AMERCURY_BADGE_TS.PAYMENTSTATUS,
        AMERCURY_BADGE_TS.PHOTOKEY,
        AMERCURY_BADGE_TS.PHOTOSOURCE,
        AMERCURY_BADGE_TS.PHOTOSOURCETYPE,
        ROW_NUMBER() OVER (
            PARTITION BY AMERCURY_BADGE_TS.ID
            ORDER BY AMERCURY_BADGE_TS.INT_INSERT_DATE DESC, AMERCURY_BADGE_TS.VERSIONNUMBER DESC
        ) AS rn
    FROM
        workspace.prxbi_ts.wc_mercury_badge_ts AMERCURY_BADGE_TS
    WHERE
        AMERCURY_BADGE_TS.INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
        AND AMERCURY_BADGE_TS.INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
) AS deduped_source
WHERE rn = 1;

In [ ]:
-- MAGIC %sql
-- Verify count of records in staging table
SELECT COUNT(*) AS c_badge_staging_count
FROM workspace.prxbi_dw.c_badge_staging;

## Flow Table (I$)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {60}: Removed DBMS_STATS for staging table (not applicable)
-- SCEN_TASK_NO {70}: (No-op)
-- SCEN_TASK_NO {80}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {90}: Create flow table
CREATE TABLE workspace.prxbi_dw.i_wc_badge_details_d_flow (
	ROW_WID		BIGINT,
	BADGE_ID		STRING,
	BADGE_LOCATION		STRING,
	BADGE_TOKEN		STRING,
	BADGE_VERSION		BIGINT,
	CONTACT_EMAIL		STRING,
	CONTACT_FIRST_NAME		STRING,
	CONTACT_LAST_NAME		STRING,
	CONTACT_JOB_TITLE		STRING,
	CONTACT_PERSON_ID		STRING,
	CREATION_REG_TYPE		STRING,
	CREATION_TYPE		STRING,
	CULTURE		STRING,
	CUSTOMER_TYPE		STRING,
	EVENT_EDITION_CODE		STRING,
	BADGE_UPDATE_FLG		STRING,
	MARKETING_PREF_PROMPT		STRING,
	ORG_NAME		STRING,
	ORG_CITY		STRING,
	ORG_COUNTRY		STRING,
	ORG_ID		STRING,
	ORG_STATE		STRING,
	PARTICIPATING_ORG_ID		STRING,
	PRODUCT_CODE		STRING,
	QR_CODE		STRING,
	REGISTRATION_ID		STRING,
	STATUS		BIGINT,
	STAFF_COMPANY_NAME		STRING,
	STAFF_COMPANY_ADDR		STRING,
	STAFF_PHONE_NUM		STRING,
	STAFF_REPORTING		STRING,
	STAFF_STANDS		STRING,
	STAFF_USER_ACCESS		STRING,
	VERSION_NUM		BIGINT,
	INTEGRATION_ID		STRING,
	DATASOURCE_NUM_ID		STRING,
	W_INSERT_DT		TIMESTAMP,
	W_UPDATE_DT		TIMESTAMP,
	MOBILEPHONE		STRING,
	FIRSTSCANNEDDATE		TIMESTAMP,
	LASTPRINTEDDATE		TIMESTAMP,
	FIRSTSCANNEDDATE_FLG		STRING,
	LASTPRINTEDDATE_FLG		STRING,
	ACCESSVALIDITYMODIFIEDDATE		TIMESTAMP,
	CREATEDDATE		TIMESTAMP,
	COMPANYPRODUCTCODE		STRING,
	PAYMENTSTATUS		STRING,
	PHOTOKEY		STRING,
	PHOTOSOURCE		STRING,
	PHOTOSOURCETYPE		STRING,
	PACKAGE_NAME		STRING,
	IND_UPDATE		STRING
) USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {100}: Insert into flow table (without initial NOT EXISTS filter)
INSERT INTO workspace.prxbi_dw.i_wc_badge_details_d_flow
(
	BADGE_ID,
	BADGE_LOCATION,
	BADGE_TOKEN,
	BADGE_VERSION,
	CONTACT_EMAIL,
	CONTACT_FIRST_NAME,
	CONTACT_LAST_NAME,
	CONTACT_JOB_TITLE,
	CONTACT_PERSON_ID,
	CREATION_REG_TYPE,
	CREATION_TYPE,
	CULTURE,
	CUSTOMER_TYPE,
	EVENT_EDITION_CODE,
	BADGE_UPDATE_FLG,
	MARKETING_PREF_PROMPT,
	ORG_NAME,
	ORG_CITY,
	ORG_COUNTRY,
	ORG_ID,
	ORG_STATE,
	PARTICIPATING_ORG_ID,
	PRODUCT_CODE,
	QR_CODE,
	REGISTRATION_ID,
	STATUS,
	STAFF_COMPANY_NAME,
	STAFF_COMPANY_ADDR,
	STAFF_PHONE_NUM,
	STAFF_REPORTING,
	STAFF_STANDS,
	STAFF_USER_ACCESS,
	VERSION_NUM,
	INTEGRATION_ID,
	DATASOURCE_NUM_ID,
	MOBILEPHONE,
	FIRSTSCANNEDDATE,
	LASTPRINTEDDATE,
	FIRSTSCANNEDDATE_FLG,
	LASTPRINTEDDATE_FLG,
	ACCESSVALIDITYMODIFIEDDATE,
	CREATEDDATE,
	COMPANYPRODUCTCODE,
	PAYMENTSTATUS,
	PHOTOKEY,
	PHOTOSOURCE,
	PHOTOSOURCETYPE,
	PACKAGE_NAME,
	IND_UPDATE
)
SELECT 
    JOIN1_A.ID AS BADGE_ID,
	JOIN1_A.BADGELOCATION AS BADGE_LOCATION,
	JOIN1_A.BADGETOKEN AS BADGE_TOKEN,
	JOIN1_A.BADGEVERSION AS BADGE_VERSION,
	JOIN1_A.CONTACTEMAIL AS CONTACT_EMAIL,
	JOIN1_A.CONTACTFIRSTNAME AS CONTACT_FIRST_NAME,
	JOIN1_A.CONTACTLASTNAME AS CONTACT_LAST_NAME,
	JOIN1_A.CONTACTJOBTITLE AS CONTACT_JOB_TITLE,
	JOIN1_A.CONTACTPERSONRXMASTERID AS CONTACT_PERSON_ID,
	JOIN1_A.CREATEDBYREGISTRATIONTYPE AS CREATION_REG_TYPE,
	JOIN1_A.CREATEDBYTYPE AS CREATION_TYPE,
	JOIN1_A.CULTURE AS CULTURE,
	JOIN1_A.CUSTOMERTYPE AS CUSTOMER_TYPE,
	JOIN1_A.EVENTEDITIONGBSCODE AS EVENT_EDITION_CODE,
	JOIN1_A.ISBADGEUPDATE AS BADGE_UPDATE_FLG,
	JOIN1_A.MARKETINGPREFERENCESPROMPTREQU AS MARKETING_PREF_PROMPT,
	JOIN1_A.ORGANISATIONDISPLAYNAME AS ORG_NAME,
	JOIN1_A.ORGANISATIONCITY AS ORG_CITY,
	JOIN1_A.ORGANISATIONCOUNTRYCODE AS ORG_COUNTRY,
	JOIN1_A.ORGANISATIONRXMASTERID AS ORG_ID,
	JOIN1_A.ORGANISATIONSTATE AS ORG_STATE,
	JOIN1_A.PARTICIPATINGORGANISATIONID AS PARTICIPATING_ORG_ID,
	JOIN1_A.PRODUCTCODE AS PRODUCT_CODE,
	JOIN1_A.QRCODECONTENT AS QR_CODE,
	JOIN1_A.REGISTRATIONID AS REGISTRATION_ID,
	JOIN1_A.STATUS AS STATUS,
	JOIN1_A.SUPPORTSTAFFCOMPANYNAME AS STAFF_COMPANY_NAME,
	JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS AS STAFF_COMPANY_ADDR,
	JOIN1_A.SUPPORTSTAFFMOBILEPHONE AS STAFF_PHONE_NUM,
	JOIN1_A.SUPPORTSTAFFREPORTSTO AS STAFF_REPORTING,
	JOIN1_A.SUPPORTSTAFFSTANDS AS STAFF_STANDS,
	JOIN1_A.SUPPORTSTAFFUSERACCESS AS STAFF_USER_ACCESS,
	JOIN1_A.VERSIONNUMBER AS VERSION_NUM,
	JOIN1_A.ID AS INTEGRATION_ID,
	CAST(${DATASOURCE_NUM_ID} AS STRING) AS DATASOURCE_NUM_ID,
	JOIN1_A.MOBILEPHONE AS MOBILEPHONE,
	JOIN1_A.FIRSTSCANNEDDATE AS FIRSTSCANNEDDATE,
	JOIN1_A.LASTPRINTEDDATE AS LASTPRINTEDDATE,
	CASE WHEN JOIN1_A.FIRSTSCANNEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS FIRSTSCANNEDDATE_FLG,
	CASE WHEN JOIN1_A.LASTPRINTEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS LASTPRINTEDDATE_FLG,
	JOIN1_A.ACCESSVALIDITYMODIFIEDDATE AS ACCESSVALIDITYMODIFIEDDATE,
	JOIN1_A.CREATEDDATE AS CREATEDDATE,
	JOIN1_A.COMPANYPRODUCTCODE AS COMPANYPRODUCTCODE,
	JOIN1_A.PAYMENTSTATUS AS PAYMENTSTATUS,
	JOIN1_A.PHOTOKEY AS PHOTOKEY,
	JOIN1_A.PHOTOSOURCE AS PHOTOSOURCE,
	JOIN1_A.PHOTOSOURCETYPE AS PHOTOSOURCETYPE,
	WC_BADGE_PRODUCT_D_2.NAME_1 AS PACKAGE_NAME,
	'I' AS IND_UPDATE
FROM	workspace.prxbi_dw.c_badge_staging JOIN1_A
LEFT OUTER JOIN  (
    SELECT 
        WC_BADGE_PRODUCT_D_1.ID AS ID,
        WC_BADGE_PRODUCT_D_1.SKU AS SKU,
        WC_BADGE_PRODUCT_D_1.NAME AS NAME,
        WC_BADGE_PRODUCT_D_1.COL AS COL,
        WC_BADGE_PRODUCT_D_1.SKU_1 AS SKU_1,
        WC_BADGE_PRODUCT_D_1.NAME_1 AS NAME_1   
    FROM 
        (
        SELECT 
            WC_BADGE_PRODUCT_D.ID AS ID,
            WC_BADGE_PRODUCT_D.SKU AS SKU,
            WC_BADGE_PRODUCT_D.NAME AS NAME,
            rank() over(partition by WC_BADGE_PRODUCT_D.SKU order by WC_BADGE_PRODUCT_D.ID desc) AS COL,
            WC_BADGE_PRODUCT_D.SKU AS SKU_1,
            WC_BADGE_PRODUCT_D.NAME AS NAME_1   
        FROM 
            workspace.prxbi_dw.wc_badge_product_d WC_BADGE_PRODUCT_D  
        ) WC_BADGE_PRODUCT_D_1  
    WHERE
        WC_BADGE_PRODUCT_D_1.COL = 1
) WC_BADGE_PRODUCT_D_2
ON  JOIN1_A.PRODUCTCODE = WC_BADGE_PRODUCT_D_2.SKU_1;

In [ ]:
-- MAGIC %sql
-- Verify count of records in flow table
SELECT COUNT(*) AS i_wc_badge_details_d_flow_count
FROM workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {110}:
Create index (converted to
OPTIMIZE ZORDER BY)
-- Disable
ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_d_flow
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {120}: Removed DBMS_STATS for flow table (not applicable)
SELECT 1 = 1;

-- No-op to preserve task number

## Mark Records for Update

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {130}: Mark existing records in flow table for update (F.10 rewrite)
MERGE INTO workspace.prxbi_dw.i_wc_badge_details_d_flow AS T
USING (
    SELECT
        INTEGRATION_ID,
        DATASOURCE_NUM_ID
    FROM
        workspace.prxbi_dw.wc_badge_details_d
) AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

## MERGE into Target (WC_BADGE_DETAILS_D)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {140}: Command skipped due to chosen DETECTION_STRATEGY (No-op)
-- SCEN_TASK_NO {150} and {160}: Combined UPDATE and INSERT into a single MERGE statement (F.3, F.10 rewrite)
-- ROW_WID is assumed to be GENERATED ALWAYS AS IDENTITY and is excluded from INSERT lists.
MERGE INTO workspace.prxbi_dw.wc_badge_details_d AS T
USING workspace.prxbi_dw.i_wc_badge_details_d_flow AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
	T.BADGE_ID = S.BADGE_ID,
	T.BADGE_LOCATION = S.BADGE_LOCATION,
	T.BADGE_TOKEN = S.BADGE_TOKEN,
	T.BADGE_VERSION = S.BADGE_VERSION,
	T.CONTACT_EMAIL = S.CONTACT_EMAIL,
	T.CONTACT_FIRST_NAME = S.CONTACT_FIRST_NAME,
	T.CONTACT_LAST_NAME = S.CONTACT_LAST_NAME,
	T.CONTACT_JOB_TITLE = S.CONTACT_JOB_TITLE,
	T.CONTACT_PERSON_ID = S.CONTACT_PERSON_ID,
	T.CREATION_REG_TYPE = S.CREATION_REG_TYPE,
	T.CREATION_TYPE = S.CREATION_TYPE,
	T.CULTURE = S.CULTURE,
	T.CUSTOMER_TYPE = S.CUSTOMER_TYPE,
	T.EVENT_EDITION_CODE = S.EVENT_EDITION_CODE,
	T.BADGE_UPDATE_FLG = S.BADGE_UPDATE_FLG,
	T.MARKETING_PREF_PROMPT = S.MARKETING_PREF_PROMPT,
	T.ORG_NAME = S.ORG_NAME,
	T.ORG_CITY = S.ORG_CITY,
	T.ORG_COUNTRY = S.ORG_COUNTRY,
	T.ORG_ID = S.ORG_ID,
	T.ORG_STATE = S.ORG_STATE,
	T.PARTICIPATING_ORG_ID = S.PARTICIPATING_ORG_ID,
	T.PRODUCT_CODE = S.PRODUCT_CODE,
	T.QR_CODE = S.QR_CODE,
	T.REGISTRATION_ID = S.REGISTRATION_ID,
	T.STATUS = S.STATUS,
	T.STAFF_COMPANY_NAME = S.STAFF_COMPANY_NAME,
	T.STAFF_COMPANY_ADDR = S.STAFF_COMPANY_ADDR,
	T.STAFF_PHONE_NUM = S.STAFF_PHONE_NUM,
	T.STAFF_REPORTING = S.STAFF_REPORTING,
	T.STAFF_STANDS = S.STAFF_STANDS,
	T.STAFF_USER_ACCESS = S.STAFF_USER_ACCESS,
	T.VERSION_NUM = S.VERSION_NUM,
	T.MOBILEPHONE = S.MOBILEPHONE,
	T.FIRSTSCANNEDDATE = S.FIRSTSCANNEDDATE,
	T.LASTPRINTEDDATE = S.LASTPRINTEDDATE,
	T.FIRSTSCANNEDDATE_FLG = S.FIRSTSCANNEDDATE_FLG,
	T.LASTPRINTEDDATE_FLG = S.LASTPRINTEDDATE_FLG,
	T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE,
	T.CREATEDDATE = S.CREATEDDATE,
	T.COMPANYPRODUCTCODE = S.COMPANYPRODUCTCODE,
	T.PAYMENTSTATUS = S.PAYMENTSTATUS,
	T.PHOTOKEY = S.PHOTOKEY,
	T.PHOTOSOURCE = S.PHOTOSOURCE,
	T.PHOTOSOURCETYPE = S.PHOTOSOURCETYPE,
	T.PACKAGE_NAME = S.PACKAGE_NAME,
	T.W_UPDATE_DT = current_timestamp()
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
	BADGE_ID,
	BADGE_LOCATION,
	BADGE_TOKEN,
	BADGE_VERSION,
	CONTACT_EMAIL,
	CONTACT_FIRST_NAME,
	CONTACT_LAST_NAME,
	CONTACT_JOB_TITLE,
	CONTACT_PERSON_ID,
	CREATION_REG_TYPE,
	CREATION_TYPE,
	CULTURE,
	CUSTOMER_TYPE,
	EVENT_EDITION_CODE,
	BADGE_UPDATE_FLG,
	MARKETING_PREF_PROMPT,
	ORG_NAME,
	ORG_CITY,
	ORG_COUNTRY,
	ORG_ID,
	ORG_STATE,
	PARTICIPATING_ORG_ID,
	PRODUCT_CODE,
	QR_CODE,
	REGISTRATION_ID,
	STATUS,
	STAFF_COMPANY_NAME,
	STAFF_COMPANY_ADDR,
	STAFF_PHONE_NUM,
	STAFF_REPORTING,
	STAFF_STANDS,
	STAFF_USER_ACCESS,
	VERSION_NUM,
	INTEGRATION_ID,
	DATASOURCE_NUM_ID,
	MOBILEPHONE,
	FIRSTSCANNEDDATE,
	LASTPRINTEDDATE,
	FIRSTSCANNEDDATE_FLG,
	LASTPRINTEDDATE_FLG,
	ACCESSVALIDITYMODIFIEDDATE,
	CREATEDDATE,
	COMPANYPRODUCTCODE,
	PAYMENTSTATUS,
	PHOTOKEY,
	PHOTOSOURCE,
	PHOTOSOURCETYPE,
	PACKAGE_NAME,
	W_INSERT_DT,
	W_UPDATE_DT
) VALUES (
	S.BADGE_ID,
	S.BADGE_LOCATION,
	S.BADGE_TOKEN,
	S.BADGE_VERSION,
	S.CONTACT_EMAIL,
	S.CONTACT_FIRST_NAME,
	S.CONTACT_LAST_NAME,
	S.CONTACT_JOB_TITLE,
	S.CONTACT_PERSON_ID,
	S.CREATION_REG_TYPE,
	S.CREATION_TYPE,
	S.CULTURE,
	S.CUSTOMER_TYPE,
	S.EVENT_EDITION_CODE,
	S.BADGE_UPDATE_FLG,
	S.MARKETING_PREF_PROMPT,
	S.ORG_NAME,
	S.ORG_CITY,
	S.ORG_COUNTRY,
	S.ORG_ID,
	S.ORG_STATE,
	S.PARTICIPATING_ORG_ID,
	S.PRODUCT_CODE,
	S.QR_CODE,
	S.REGISTRATION_ID,
	S.STATUS,
	S.STAFF_COMPANY_NAME,
	S.STAFF_COMPANY_ADDR,
	S.STAFF_PHONE_NUM,
	S.STAFF_REPORTING,
	S.STAFF_STANDS,
	S.STAFF_USER_ACCESS,
	S.VERSION_NUM,
	S.INTEGRATION_ID,
	S.DATASOURCE_NUM_ID,
	S.MOBILEPHONE,
	S.FIRSTSCANNEDDATE,
	S.LASTPRINTEDDATE,
	S.FIRSTSCANNEDDATE_FLG,
	S.LASTPRINTEDDATE_FLG,
	S.ACCESSVALIDITYMODIFIEDDATE,
	S.CREATEDDATE,
	S.COMPANYPRODUCTCODE,
	S.PAYMENTSTATUS,
	S.PHOTOKEY,
	S.PHOTOSOURCE,
	S.PHOTOSOURCETYPE,
	S.PACKAGE_NAME,
	current_timestamp(),
	current_timestamp()
);

## Cleanup

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {170}: Removed COMMIT (implicit in Databricks Delta)
-- SCEN_TASK_NO {180}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {190}: (No-op)
-- SCEN_TASK_NO {200}: (No-op)
-- SCEN_TASK_NO {210}: Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_badge_staging;

## Validation

In [ ]:
-- MAGIC %sql
-- Final record count for target table
SELECT COUNT(*) AS final_target_table_count
FROM workspace.prxbi_dw.wc_badge_details_d;

In [ ]:
-- MAGIC %sql
-- Sample recently updated records
SELECT * FROM workspace.prxbi_dw.wc_badge_details_d
WHERE W_UPDATE_DT >= current_date() - INTERVAL '1' DAY
ORDER BY W_UPDATE_DT DESC
LIMIT 100;

## Conversion Notes and Manual Actions Required

- **`ROW_WID` Column**: The `WC_BADGE_DETAILS_D_SEQ.NEXTVAL` from the original INSERT statement has been removed. It is assumed that the `ROW_WID` column in `workspace.prxbi_dw.wc_badge_details_d` is configured as `BIGINT GENERATED ALWAYS AS IDENTITY` in the table's DDL definition. If this is not the case, manual intervention is required to define `ROW_WID` as an identity column or to provide a suitable unique identifier generation mechanism.
- **ETL Parameter Table (`wc_etl_parameters`)**: This script relies on `workspace.prxbi_dw.wc_etl_parameters` existing and containing `etl_last_extract_time`, `etl_current_extract_time`, and `row_wid` for the specified `ETL_JOB_TYPE`.
- **`DATASOURCE_NUM_ID`**: Hardcoded value `380` has been replaced with `${DATASOURCE_NUM_ID}` widget. Ensure this widget is correctly passed.
- **Optimized ZORDER BY**: `CREATE INDEX` statements are converted to `OPTIMIZE ... ZORDER BY` to maintain data clustering for performance. A `SET` command is included before each `OPTIMIZE` to disable stats collection checks, preventing potential errors or warnings (`DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS`). This setting is session-level and transient.
- **Missing Error Handling**: ODI often includes specific error logging. This conversion focuses on SQL logic; comprehensive error handling (e.g., to E$ tables) would need to be added separately using Databricks' error handling patterns (e.g., `spark.sql.execution.error.storage.path`, `Delta Live Tables` error handling, or custom logic).
- **Performance Tuning**: Initial loading into staging/flow tables could benefit from `INSERT OVERWRITE` or `COPY INTO` patterns for full loads, if applicable, based on volume and frequency. This script uses `CREATE OR REPLACE TABLE` and `INSERT INTO` which are generally robust.